# 🚀 Step 2: Train Unified Gurukul Lite V2

**Purpose:** Train on pre-tokenized data

- **Input:** `tokenized_data.zip` (from tokenization notebook)
- **Output:** `gurukul_lite_v2` adapter
- **Time:** 4-6 hours (depends on data size)

---

## Prerequisites

✅ You completed the tokenization notebook
✅ You downloaded `tokenized_data.zip`

---

## Instructions

1. Enable **T4 GPU**
2. Upload `tokenized_data.zip`
3. Run all cells
4. Keep session alive (F12 → paste keep-alive code)
5. Download trained adapter after 4-6 hours

## 1️⃣ Check GPU

In [ ]:
!nvidia-smi

## 2️⃣ Install Dependencies

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets

## 3️⃣ Upload & Extract Tokenized Data

Upload `tokenized_data.zip` using Files panel (📁)

In [ ]:
import os

# Check for tokenized data
if os.path.exists('tokenized_data.zip'):
    print("📦 Found tokenized_data.zip")
    print("🔄 Extracting...")
    !unzip -q tokenized_data.zip
    print("✅ Extraction complete!")
    
    print("\n📁 Contents:")
    !ls -lh tokenized_data/
else:
    print("❌ tokenized_data.zip not found!")
    print("Please upload it using the Files panel")

## 4️⃣ Load Pre-Tokenized Data

Much faster than tokenizing from scratch!

In [ ]:
from datasets import load_from_disk
from transformers import AutoTokenizer

print("📥 Loading pre-tokenized data...")

# Load datasets
train_data = load_from_disk("tokenized_data/train")
val_data = load_from_disk("tokenized_data/val")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("tokenized_data/tokenizer")

print(f"\n✅ Data loaded!")
print(f"   📊 Train: {len(train_data):,} samples")
print(f"   📊 Val:   {len(val_data):,} samples")
print(f"   🔤 Tokenizer: {tokenizer.name_or_path}")

## 5️⃣ Load Base Model (8-bit)

In [ ]:
from transformers import AutoModelForCausalLM
from peft import prepare_model_for_kbit_training
import torch

BASE_MODEL = "bigscience/bloomz-560m"

print(f"📥 Loading base model: {BASE_MODEL}")
print("   (Using 8-bit quantization to save GPU memory)")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    load_in_8bit=True,
    device_map="auto",
    torch_dtype=torch.float16
)

# Prepare model for 8-bit training with LoRA
print("🔧 Preparing model for 8-bit training...")
model = prepare_model_for_kbit_training(model)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

print(f"✅ Model loaded and prepared for training")

## 6️⃣ Configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

print("⚙️  Configuring LoRA...")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query_key_value"],
    bias="none"
)

model = get_peft_model(model, lora_config)

print("✅ LoRA configured (r=8, alpha=16)")
print("\n📊 Trainable parameters:")
model.print_trainable_parameters()

## 7️⃣ Setup Training Configuration

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = "gurukul_lite_v2"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=200,
    logging_steps=100,
    save_steps=1000,
    eval_steps=1000,
    evaluation_strategy="steps",
    save_total_limit=3,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    report_to="none",
    gradient_checkpointing=True,
    optim="adamw_torch"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator
)

print("✅ Training configured")
print(f"   Epochs: 3")
print(f"   Batch size: 4 (effective: 16 with grad accumulation)")
print(f"   Total steps: ~{len(train_data) // 16 * 3:,}")

## 8️⃣ Start Training

⚠️ **IMPORTANT:** Run the keep-alive code now!

Press **F12** → Console → Paste:

```javascript
function KeepClicking(){
  console.log("Clicking");
  document.querySelector("colab-connect-button").click()
}
setInterval(KeepClicking, 60000)
```

In [ ]:
from datetime import datetime

print("="*80)
print("🚀 STARTING TRAINING")
print("="*80)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Training on {len(train_data):,} samples")
print("="*80 + "\n")

start_time = datetime.now()

# Train!
trainer.train()

end_time = datetime.now()
duration = (end_time - start_time).total_seconds() / 3600

print("\n" + "="*80)
print("✅ TRAINING COMPLETE!")
print("="*80)
print(f"Duration: {duration:.2f} hours")
print(f"Finished: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

## 9️⃣ Save Final Adapter

In [ ]:
print("💾 Saving final adapter...")

# Save adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ Adapter saved to {OUTPUT_DIR}/")
print(f"\n📁 Contents:")
!ls -lh {OUTPUT_DIR}/

## 🔟 Create Download ZIP

In [ ]:
print("📦 Creating download package...")

!zip -r -q gurukul_lite_v2.zip gurukul_lite_v2/

import os
size_mb = os.path.getsize('gurukul_lite_v2.zip') / (1024 * 1024)

print(f"\n✅ Created: gurukul_lite_v2.zip ({size_mb:.1f} MB)")
print("\n📥 Download instructions:")
print("   1. Click Files panel (📁) on the left")
print("   2. Right-click 'gurukul_lite_v2.zip'")
print("   3. Select 'Download'")
print("   4. Extract to: C:\\pc\\Project\\adapters\\gurukul_lite_v2\\")

## 1️⃣1️⃣ Test the Adapter (Optional)

In [ ]:
from peft import PeftModel

print("🧪 Testing adapter...\n")

# Load base model
test_model = AutoModelForCausalLM.from_pretrained(
    "bigscience/bloomz-560m",
    device_map="auto",
    torch_dtype=torch.float16
)

# Load adapter
test_model = PeftModel.from_pretrained(test_model, OUTPUT_DIR)
test_model.eval()

# Test prompts
test_prompts = [
    "नमस्ते, मेरा नाम",  # Hindi
    "ਸਤ ਸ੍ਰੀ ਅਕਾਲ, ਮੇਰਾ ਨਾਮ",  # Punjabi
    "সুপ্রভাত, আমার নাম"  # Bengali
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(test_model.device)
    outputs = test_model.generate(**inputs, max_length=50, do_sample=True, temperature=0.7)
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Output: {generated}")
    print("-" * 80)

---

## ✅ **TRAINING COMPLETE!**

### Next Steps:

1. ✅ Download `gurukul_lite_v2.zip`
2. ✅ Extract to `C:\pc\Project\adapters\gurukul_lite_v2\`
3. ✅ Test with your API
4. ✅ Create PR and notify Task Bank

---

### Congratulations! 🎉

You've successfully trained a unified adapter supporting **38 languages**!
